<div style="width:100%; text-align:center; padding:10px 0;">
<img src="project_header.png" style="width:100%; max-width:100vw; height:auto; display:block; margin:0 auto;">
</div>

# EO Africa SWAM project 
## Batch-process C2X-Complex Atmospheric correction

#### This notebook does the following:
* reads in the Sentinel-2 L1C filenames from the textfile
* Run a SNAP graph processing framework ("resample_subset_idepix_c2xc_**LAKE**.xml") on each file
* **LAKE** options are CW, TW, MV, VV, ZV, RV (default example is TW)     
* the graph does the following steps: 
<br> 1. read the file, 
<br> 2. resample the radiance to 60m (can be changed to 20m or 10m in the xml file)
<br> 3. subset to the region of interest (defined in xml file)
<br> 4. apply the IdePix processor
<br> 5. bandmerge the IdePix classification output
<br> 6. apply the C2X-Complex atmospheric correction processor
<br> 7. save the output as netcdf

#### Requirements 
* Make sure that you have already created a "**LAKE**_filenames.txt" with the *TSM_Step1_Create_list_of_filenames.ipynb* notebook

#### Settings to adjust manually:
* In code cell 1 define the lake/dam name (options: ZV, RV, TW, MV, VV, CW), default is TW.
* In code cell 4 define the path to the bin folder of your SNAP installation
* If changing to another lake (other than TW) save a copy of "resample_subset_idepix_c2xc_**LAKE**.xml" and adjust the polygon in line 25 and spatial resolution in line 10

### Version history:
* Version 1.0, 23 Feb, 2026

##### Authors:
**Marie Smith**, CSIR, South Africa

In [ ]:
# VERY IMPORTANT - Make sure to select the appropriate lake here (options: CW, MV, RV, VV, TW, ZV)
lake_name = "TW"                 

In [ ]:
import os
import glob
import subprocess
import numpy as np
import re
import time
import warnings
warnings.filterwarnings('ignore')

Read in the textfile containing the filenames that you want to process:

In [ ]:
with open(lake_name+'_filenames.txt', 'r') as file:
    filenames = file.read().splitlines()
num = len(filenames)
print(str(num) + ' filtered files total')  

Define the file paths and folders to use during the processing steps:

In [ ]:
## the file path and name of the xml graph:
gpt_graph = "resample_subset_idepix_c2xc_"+lake_name+".xml"    

## the path to the folder where completed files will be saved
if not os.path.exists(lake_name):
    os.makedirs(lake_name)
    print(f"Created folder: {lake_name}")
else:
    print(f"Folder {lake_name} already exists.") [web:1][web:5]
output_path = "/"+lake_name+"/"       

## the path to the "bin" folder in your snap installation:
snap_gpt = '/usr/local/snap/bin/gpt'

Here we define the functions that we will use:

In [ ]:
def pre_processing(in_image, lake_name,output_path):
    image_info = os.path.basename(in_image).split('_')   
    out_image = output_path+image_info[0]+'_'+image_info[2]+'_'+image_info[5]+'_C2XC_'+lake_name
    
    subprocess.run([
    snap_gpt,
    gpt_graph,
    '-SsourceProduct=',in_image,
    '-t',out_image,
    '-f','NetCDF4-BEAM'  #'NetCDF4-CF'
    ])

# Function to extract unique identifier from original filename
def extract_id_from_original(path):
    # Extract filename from path
    filename = os.path.basename(os.path.dirname(path))  # Gets the folder with SAFE suffix
    # filename example: "S2A_MSIL1C_20150810T084916_N0500_R121_T34HBJ_20231016T203917.SAFE"
    # Extract token up to tile code, here let's extract first 7 underscore parts for simplicity
    parts = filename.split('_')
    if len(parts) >= 7:
        return f"{parts[0]}_{parts[2]}_{parts[5]}" # e.g. S2A_20151019T082012_T34HBJ
    return None

# Function to extract unique identifier from processed filename
def extract_id_from_processed(path):
    filename = os.path.basename(path)
    # filename example: "S2A_20151019T082012_T34HBJ_C2XC_MV.nc"
    parts = filename.split('_')
    if len(parts) >= 3:
        return f"{parts[0]}_{parts[1]}_{parts[2]}" # e.g. S2A_20151019T082012_T34HBJ
    return None    

Sometimes the processing or connection cuts out before the entire list of files can be processed. This next section checks whether you have already processed any files for that dam by comparing the dates from the list of filenames to the list of processed files. If you have already completed some, it picks up where you left off

In [ ]:
# Check if there are already any data in the output folder
processed_files = [os.path.join(output_path, f) for f in os.listdir(output_path) if f.endswith('.nc')]

# Get sets of identifiers
original_ids = set(filter(None, [extract_id_from_original(f) for f in filenames]))
processed_ids = set(filter(None, [extract_id_from_processed(f) for f in processed_files]))

# Remaining IDs to process
remaining_ids = original_ids - processed_ids

# Filter original files list to only keep files that match remaining IDs
remaining_files = [f for f in filenames if extract_id_from_original(f) in remaining_ids]

print(f"Remaining files to process: {len(remaining_files)}")
# Then process these remaining_files as needed

Below is the main processing step where you read the input file location, run the file through the SNAP using the graph processing framework, and create a netcdf output. For a single file this step can take a few minutes to complete for a small lake, but it can take over 30 minutes to process a scene over a larger lake - so be patient!

In [ ]:
# process each satellite image
for i in range(0,num):
    start_time = time.time()  # Record the start time of the current iteration
    one_image = remaining_files[i]
    dirpath = os.path.dirname(one_image)
    print('Processing >> ',one_image)
    pre_processing(dirpath, lake_name,output_path)
    print('Finish >>',one_image)
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Iteration {i+1}: Elapsed time = {elapsed_time:.4f} seconds")
    print('-------------------------------------------------------------')

print('***************** all finished, well done ^_^ *******************')